In [27]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/rjmanoj/credit-card-customer-churn-prediction/Churn_Modelling.csv


In [28]:
df=pd.read_csv('/kaggle/input/datasets/rjmanoj/credit-card-customer-churn-prediction/Churn_Modelling.csv'
)

In [29]:
pd.set_option('display.max_rows',None)
pd.set_option('display.max_columns',None)

In [30]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [31]:
df.isnull().sum()*100/df.shape[0]

RowNumber          0.0
CustomerId         0.0
Surname            0.0
CreditScore        0.0
Geography          0.0
Gender             0.0
Age                0.0
Tenure             0.0
Balance            0.0
NumOfProducts      0.0
HasCrCard          0.0
IsActiveMember     0.0
EstimatedSalary    0.0
Exited             0.0
dtype: float64

In [32]:
df_num_clm=df.select_dtypes(include=['int','float'])

In [33]:
df_num_clm.head()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,619,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,608,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,502,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,699,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,850,43,2,125510.82,1,1,1,79084.10,0


In [34]:
df_cat_clm=df.select_dtypes(include=['object'])

In [35]:
df_cat_clm.head()

,Surname,Geography,Gender
0,Hargrave,France,Female
1,Hill,Spain,Female
2,Onio,France,Female
3,Boni,France,Female
4,Mitchell,Spain,Female


In [36]:
df_num_clm.isnull().sum().sum()

np.int64(0)

In [37]:
for i in df_num_clm:
    df_num_clm[i]=df_num_clm[i].replace(np.nan,df_num_clm[i].median())

In [38]:
df_cat_clm.isnull().sum().sum()

np.int64(0)

In [39]:
for i in df_cat_clm:
    df_cat_clm[i]=df_cat_clm[i].replace(np.nan,df_cat_clm[i].mode()[0])

In [40]:
freq_encoding=['Surname', 'Geography']
one_hot_encoding=['Gender']

In [47]:
for i in freq_encoding:
    s=df_cat_clm[i].value_counts()
    df_cat_clm[i]=df_cat_clm[i].map(s)

In [52]:
df_cat_clm['Gender']=pd.DataFrame(pd.get_dummies(df_cat_clm['Gender'],dtype=int, drop_first=True))

In [53]:
df_new=pd.concat([df_num_clm,df_cat_clm],axis=1)

In [54]:
df_new.head()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Surname,Geography,Gender
0,1,15634602,619,42,2,0.00,1,1,1,101348.88,1,1558,5014,0
1,2,15647311,608,41,1,83807.86,1,0,1,112542.58,0,391,2477,0
2,3,15619304,502,42,8,159660.80,3,1,0,113931.57,1,344,5014,0
3,4,15701354,699,39,1,0.00,2,0,0,93826.63,0,350,5014,0
4,5,15737888,850,43,2,125510.82,1,1,1,79084.10,0,300,2477,0


In [55]:
from sklearn.preprocessing import StandardScaler

In [56]:
ss=StandardScaler()

In [57]:
ss.fit(df_new)

StandardScaler()

In [58]:
df_new.describe()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Surname,Geography,Gender
count,10000.00000,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,5000.50000,1.569094e+07,650.528800,38.921800,5.012800,76485.889288,1.530200,0.70550,0.515100,100090.239881,0.203700,622.050200,3757.080600,0.545700
std,2886.89568,7.193619e+04,96.653299,10.487806,2.892174,62397.405202,0.581654,0.45584,0.499797,57510.492818,0.402769,437.503346,1260.557375,0.497932
min,1.00000,1.556570e+07,350.000000,18.000000,0.000000,0.000000,1.000000,0.00000,0.000000,11.580000,0.000000,26.000000,2477.000000,0.000000
25%,2500.75000,1.562853e+07,584.000000,32.000000,3.000000,0.000000,1.000000,0.00000,0.000000,51002.110000,0.000000,344.000000,2509.000000,0.000000
50%,5000.50000,1.569074e+07,652.000000,37.000000,5.000000,97198.540000,1.000000,1.00000,1.000000,100193.915000,0.000000,480.000000,5014.000000,1.000000
75%,7500.25000,1.575323e+07,718.000000,44.000000,7.000000,127644.240000,2.000000,1.00000,1.000000,149388.247500,0.000000,708.000000,5014.000000,1.000000
max,10000.00000,1.581569e+07,850.000000,92.000000,10.000000,250898.090000,4.000000,1.00000,1.000000,199992.480000,1.000000,1558.000000,5014.000000,1.000000


In [67]:
arr=ss.transform(df_new).round(2)
arr

array([[-1.73, -0.78, -0.33, ...,  2.14,  1.  , -1.1 ],
       [-1.73, -0.61, -0.44, ..., -0.53, -1.02, -1.1 ],
       [-1.73, -1.  , -1.54, ..., -0.64,  1.  , -1.1 ],
       ...,
       [ 1.73, -1.48,  0.6 , ..., -0.32,  1.  , -1.1 ],
       [ 1.73, -0.12,  1.26, ..., -0.87, -0.99,  0.91],
       [ 1.73, -0.87,  1.46, ..., -1.36,  1.  , -1.1 ]], shape=(10000, 14))

In [69]:
df2.head()

,RowNumber,CustomerId,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Surname,Geography,Gender
0,-1.731878,-0.783213,-0.326221,0.293517,-1.041760,-1.225848,-0.911583,0.646092,0.970243,0.021886,1.977165,2.139404,0.997164,-1.095988
1,-1.731531,-0.606534,-0.440036,0.198164,-1.387538,0.117350,-0.911583,-1.547768,0.970243,0.216534,-0.505775,-0.528137,-1.015539,-1.095988
2,-1.731185,-0.995885,-1.536794,0.293517,1.032908,1.333053,2.527057,0.646092,-1.030670,0.240687,1.977165,-0.635570,0.997164,-1.095988
3,-1.730838,0.144767,0.501521,0.007457,-1.387538,-1.225848,0.807737,-1.547768,-1.030670,-0.108918,-0.505775,-0.621855,0.997164,-1.095988
4,-1.730492,0.652659,2.063884,0.388871,-1.041760,0.785728,-0.911583,0.646092,0.970243,-0.365276,-0.505775,-0.736146,-1.015539,-1.095988


In [70]:
from sklearn.model_selection import train_test_split

In [71]:
y=df2['Exited']

In [72]:
X=df2.drop(columns='Exited')

/tmp/ipykernel_58/2514133810.py:1: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  X=df2.drop(columns='Exited')


In [86]:
X_train, X_test,y_train,y_test=train_test_split(X,y, test_size=.2)

In [87]:
print(X_train.shape)

(8000, 13)


In [94]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.metrics import accuracy_score

In [95]:
model = Sequential()
model.add(Dense(3, activation='sigmoid', input_dim=13))
model.add(Dense(1, activation='sigmoid'))
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 3)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46 (184.00 B)

 Trainable params: 46 (184.00 B)

 Non-trainable params: 0 (0.00 B)

In [96]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [97]:
model.fit(X_train, y_train, epochs=5)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.0000e+00 - loss: 0.3804
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: 0.1470
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: -0.0140
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: -0.1371
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0000e+00 - loss: -0.2410


In [120]:
y_pred = model.predict(X_test).flatten()


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [127]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [126]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MAE: 0.8529531359672546
MSE: 0.9994350075721741
R2 Score: 0.052177250385284424
